<a id="dp-overview"></a>

$$
\text{动态规划 (Dynamic Programming, DP): 把大问题拆成子问题, 每个子问题只算一次, 结果存起来重复使用} \\ \quad \\
\text{能用 DP 需要两个性质:} \\
\text{1. 最优子结构 (Optimal Substructure): 大问题的最优解由子问题的最优解组成} \\
\text{2. 重叠子问题 (Overlapping Subproblems): 同一个子问题会被反复用到 (所以存起来才有意义)} \\ \quad \\
\text{贝尔曼最优性原理 (Bellman's Principle of Optimality):} \\
\text{一个最优策略不管初始状态和初始决策是什么, 剩下的决策对于第一个决策所产生的状态, 也必须构成最优策略} \\
\text{证明 (剪贴法 cut-and-paste): 设最优路径 } A \to B \to C \text{, 如果 } B \to C \text{ 这一段不是 B 到 C 的最优路径} \\
\text{把它换成更短的 } B \to C \text{ 路径, 得到一条比 "最优" 更短的 } A \to C \text{ 路径, 矛盾} \\ \quad \\
\text{例: 斐波那契数列 } F(n) = F(n-1) + F(n-2), \ F(1) = F(2) = 1 \\
\text{直接递归: 调用次数 } T(n) = T(n-1) + T(n-2) + 1 \text{, 和 F(n) 一样指数增长 } \sim \phi^n, \ \phi = \frac{1 + \sqrt{5}}{2} \approx 1.618 \\
\text{(猜 } T(n) \sim r^n \text{, 代入齐次部分 } r^2 = r + 1 \text{, 和差分方程的特征方程一样)} \\
\text{记忆化 (Memoization, 自顶向下): 算过的存进表里, 每个 F(k) 只算一次, } O(n) \\
\text{制表 (Tabulation, 自底向上): 从 F(1), F(2) 开始按顺序填表, } O(n) \text{, 不用递归}
$$

数学基础: [特征方程 (差分方程猜 λ^t)](../time_series_arima_model/time_series_arima_model.ipynb#unit-root) · [常系数线性 ODE 的特征方程](../math_basics/calculus.ipynb#ode-system)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import product
from functools import lru_cache
from scipy.optimize import milp, LinearConstraint, Bounds

np.set_printoptions(linewidth=np.inf)
np.set_printoptions(suppress=True)

def plot_nicely():
    for ax in plt.gcf().get_axes():
        ax.grid(True, which="both", alpha=0.3)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize='small')
    plt.tight_layout()

calls = {'n': 0}
def fib_naive(n):
    calls['n'] += 1
    return 1 if n <= 2 else fib_naive(n - 1) + fib_naive(n - 2)

memo_calls = {'n': 0}
@lru_cache(maxsize=None)
def fib_memo(n):
    memo_calls['n'] += 1
    return 1 if n <= 2 else fib_memo(n - 1) + fib_memo(n - 2)

def fib_table(n):
    table = [0, 1, 1] + [0] * (n - 2)
    for k in range(3, n + 1):
        table[k] = table[k - 1] + table[k - 2]
    return table[n]

phi = (1 + np.sqrt(5)) / 2
ns = list(range(5, 26))
naive_counts = []
for n in ns:
    calls['n'] = 0
    fib_naive(n)
    naive_counts.append(calls['n'])
fib_memo.cache_clear(); memo_calls['n'] = 0
print(f'F(25): 直接递归 = {fib_naive(25)}, 记忆化 = {fib_memo(25)}, 制表 = {fib_table(25)}')
print(f'F(25) 的调用次数: 直接递归 = {naive_counts[-1]}, 记忆化 = {memo_calls["n"]}')
print(f'相邻调用次数之比 = {naive_counts[-1] / naive_counts[-2]:.6f}, phi = {phi:.6f}')
# 推导: T(n) = T(n-1) + T(n-2) + 1, 令 U(n) = T(n) + 1, 则 U(n) = U(n-1) + U(n-2), U(1) = U(2) = 2, 所以 T(n) = 2F(n) - 1
print(f'T(n) == 2F(n) - 1 ? {all(t == 2 * fib_table(n) - 1 for n, t in zip(ns, naive_counts))}')

plt.figure(figsize=(4, 3))
plt.semilogy(ns, naive_counts, 'o-', label='naive recursion calls')
plt.semilogy(ns, ns, 's-', label='memo / table: n')
plt.xlabel('n')
plot_nicely()
plt.show()

<a id="knapsack-dp"></a>

$$
\text{0-1 背包的动态规划} \\
\text{状态: } dp[i][w] = \text{只考虑前 i 个物品, 容量为 w 时能得到的最大价值} \\ \quad \\
\text{状态转移方程 (推导): 第 i 个物品只有两种选择} \\
\text{不放: 容量 w 全部留给前 i - 1 个物品} \implies dp[i-1][w] \\
\text{放 (要求 } w_i \leq w \text{): 前 i - 1 个物品只能用 } w - w_i \text{ 的容量, 由最优性原理它们也要最优} \implies dp[i-1][w - w_i] + v_i \\
dp[i][w] = \max\left( dp[i-1][w], \ dp[i-1][w - w_i] + v_i \right), \quad dp[0][w] = 0 \\ \quad \\
\text{回溯 (Backtracking): 从 } dp[n][W] \text{ 往回看, 如果 } dp[i][w] \neq dp[i-1][w] \text{, 说明放了第 i 个物品, } w \leftarrow w - w_i \\ \quad \\
\text{复杂度 } O(nW) \text{ (表的大小), 比暴力的 } O(2^n) \text{ 快很多} \\
\text{但 W 是数值而不是输入的长度, 所以叫伪多项式时间 (Pseudo-polynomial); 重量不是整数时不能直接这样做}
$$

对比: [整数规划里同一个背包问题](integer_programming.ipynb#knapsack) (暴力枚举 / 分支定界 / milp 的最优价值都是 84)

In [ ]:
values_k = np.array([24, 13, 23, 15, 16, 11, 8, 26, 5, 19, 17, 9])
weights_k = np.array([12, 7, 11, 8, 9, 6, 5, 14, 3, 10, 9, 4])
capacity = 42
n_items = len(values_k)

dp = np.zeros((n_items + 1, capacity + 1), dtype=int)
for i in range(1, n_items + 1):
    wi, vi = weights_k[i - 1], values_k[i - 1]
    for w in range(capacity + 1):
        dp[i, w] = dp[i - 1, w]
        if wi <= w:
            dp[i, w] = max(dp[i, w], dp[i - 1, w - wi] + vi)

chosen, w = [], capacity
for i in range(n_items, 0, -1):
    if dp[i, w] != dp[i - 1, w]:
        chosen.append(i - 1)
        w -= weights_k[i - 1]
chosen = sorted(chosen)
print(f'DP 最优价值 = {dp[n_items, capacity]}, 选中 = {chosen}, 总重量 = {weights_k[chosen].sum()}, 总价值 = {values_k[chosen].sum()}')

m_k = milp(-values_k.astype(float), constraints=LinearConstraint(weights_k[None, :], -np.inf, capacity),
           integrality=np.ones(n_items), bounds=Bounds(0, 1))
print(f'scipy milp 最优价值 = {-m_k.fun:.0f}, 选中 = {np.flatnonzero(np.round(m_k.x)).tolist()}, 最优值一致? {np.isclose(dp[n_items, capacity], -m_k.fun)}')
print('两种方法选中的物品不同但总价值相同: 这个背包问题有不止一个最优解')
print(f'DP 表大小 = {dp.size} 个格子, 暴力枚举 = {2**n_items} 种组合')

plt.figure(figsize=(4, 3))
plt.imshow(dp, aspect='auto', origin='lower', cmap='viridis')
plt.colorbar(label='dp[i][w]')
plt.xlabel('capacity w')
plt.ylabel('items i')
plt.title('knapsack DP table')
plt.tight_layout()
plt.show()

<a id="staged-graph"></a>

$$
\text{多阶段决策: 阶段图的最短路径 (Shortest Path on a Staged Graph)} \\
\text{从起点出发, 每个阶段选一个节点, 只能从第 k 阶段走到第 k + 1 阶段, 求到终点的最短路径} \\ \quad \\
\text{逆序递推 (Backward Recursion): } f_k(s) = \text{从第 k 阶段的节点 s 到终点的最短距离} \\
f_k(s) = \min_{s' \in \text{第 k+1 阶段}} \left[ c(s, s') + f_{k+1}(s') \right], \quad f_{K}(\text{终点}) = 0 \\
\text{(由最优性原理: 从 s 出发的最短路, 去掉第一步后的那一段也必须是从 } s' \text{ 出发的最短路)} \\ \quad \\
\text{复杂度: 每个阶段 m 个节点, K 个阶段, DP 是 } O(K m^2) \text{, 枚举所有路径是 } O(m^K)
$$

In [ ]:
rng = np.random.default_rng(4)
stage_sizes = [1, 4, 4, 4, 4, 1]        # 起点, 4 个中间阶段, 终点
costs = [rng.integers(1, 20, size=(stage_sizes[k], stage_sizes[k + 1])) for k in range(len(stage_sizes) - 1)]
K = len(stage_sizes) - 1

# 逆序递推
f = [None] * (K + 1)
choice = [None] * K
f[K] = np.zeros(1, dtype=int)
for k in range(K - 1, -1, -1):
    total = costs[k] + f[k + 1][None, :]          # total[s, s'] = c(s, s') + f_{k+1}(s')
    f[k] = total.min(axis=1)
    choice[k] = total.argmin(axis=1)
path = [0]
for k in range(K):
    path.append(int(choice[k][path[-1]]))
print(f'DP 最短距离 = {f[0][0]}, 路径 (每个阶段的节点编号) = {path}')
for k in range(1, K):
    print(f'  f_{k} (第 {k} 阶段各节点到终点的最短距离) = {f[k]}')

# 暴力枚举所有路径
best_len, best_path = np.inf, None
for mids in product(*[range(s) for s in stage_sizes[1:-1]]):
    p = (0,) + mids + (0,)
    length = sum(costs[k][p[k], p[k + 1]] for k in range(K))
    if length < best_len:
        best_len, best_path = length, p
print(f'暴力枚举 {np.prod(stage_sizes[1:-1])} 条路径: 最短 = {best_len}, 路径 = {list(best_path)}, 一致? {best_len == f[0][0]}')

<a id="resource-allocation"></a>

$$
\text{资源分配问题 (Resource Allocation)} \\
\text{把 N 个单位的资源 (资金, 人员, 设备) 分给 J 个项目, 项目 j 分到 x 个单位的收益是 } g_j(x) \text{ (一张表, 不要求线性)} \\
\max \sum_{j=1}^{J} g_j(x_j) \quad \text{s.t.} \quad \sum_j x_j = N, \ x_j \in \{0, 1, \cdots, N\} \\ \quad \\
\text{状态: } F_j(n) = \text{前 j 个项目一共分 n 个单位的最大收益} \\
\text{第 j 个项目分 x 个, 剩下 n - x 个给前 j - 1 个项目 (它们也要最优):} \\
F_j(n) = \max_{0 \leq x \leq n} \left[ g_j(x) + F_{j-1}(n - x) \right], \quad F_0(n) = 0 \\
\text{复杂度 } O(J N^2) \text{; 枚举所有分配方案是 } \binom{N + J - 1}{J - 1} \text{ 种 (隔板法)}
$$

In [ ]:
from math import comb

N_units = 8
# g[j][x]: 第 j 个项目分到 x 个单位的收益 (边际收益递减, 但每个项目不一样)
g = np.array([[0, 4, 7, 9, 10, 11, 11, 11, 11],
              [0, 2, 5, 9, 12, 14, 15, 15, 15],
              [0, 5, 8, 10, 11, 12, 12, 12, 12],
              [0, 3, 6, 8, 10, 12, 13, 14, 15]])
J = len(g)

F = np.zeros((J + 1, N_units + 1), dtype=int)
alloc = np.zeros((J + 1, N_units + 1), dtype=int)
for j in range(1, J + 1):
    for n in range(N_units + 1):
        candidates = [g[j - 1][x] + F[j - 1][n - x] for x in range(n + 1)]
        alloc[j, n] = int(np.argmax(candidates))
        F[j, n] = max(candidates)

plan, n = [], N_units
for j in range(J, 0, -1):
    plan.append(int(alloc[j, n])); n -= alloc[j, n]
plan = plan[::-1]
print(f'DP: 最大收益 = {F[J, N_units]}, 分配方案 = {plan}, 各项目收益 = {[int(g[j][plan[j]]) for j in range(J)]}')

best = (-1, None); count = 0
for xs in product(range(N_units + 1), repeat=J):
    if sum(xs) != N_units: continue
    count += 1
    total = sum(g[j][xs[j]] for j in range(J))
    if total > best[0]: best = (total, xs)
print(f'暴力枚举 {count} 种方案 (隔板法 C({N_units + J - 1}, {J - 1}) = {comb(N_units + J - 1, J - 1)}): 最大收益 = {best[0]}, 一致? {best[0] == F[J, N_units]}')

<a id="dp-complexity"></a>

$$
\text{复杂度对比} \\
\begin{array}{|l|c|c|}
\hline
\text{问题} & \text{暴力} & \text{DP} \\
\hline
\text{斐波那契 F(n)} & O(\phi^n) & O(n) \\
\text{0-1 背包 (n 个物品, 容量 W)} & O(2^n) & O(nW) \\
\text{阶段图 (K 阶段, 每阶段 m 个节点)} & O(m^K) & O(K m^2) \\
\text{资源分配 (J 个项目, N 个单位)} & O(N^{J-1}) & O(J N^2) \\
\hline
\end{array} \\ \quad \\
\text{DP 的代价: 状态数可能很大 ("维数灾难" curse of dimensionality, 贝尔曼本人提出的说法)} \\
\text{例如背包问题再加一个体积约束, 状态变成 } dp[i][w][v] \text{, 表的大小乘以 V}
$$